### Importations

In [9]:
import pandas as pd

from sklearn.preprocessing import StandardScaler

### Dataset

In [2]:
df = pd.read_csv('data_student/data_labeled/X_train.csv')
df2 = pd.read_csv('data_student/data_labeled/X_test.csv')
df3 = pd.read_csv('data_student/data_labeled/y_train.csv')
df4 = pd.read_csv('data_student/data_labeled/y_test.csv')

X_learn = df.drop(['img_filename'], axis=1)
X_test = df2.drop(['img_filename'], axis=1)
y_learn = df3
y_test = df4

X_learn.head(10)

,age,blood_pressure,calcium,cholesterol,hemoglobin,height,potassium,profession,sarsaparilla,smurfberry_liquor,smurfin_donuts,vitamin_D,weight
0,161,104.55,2.53,116.47,16.55,7.80,3.98,services,high,very_high,very_low,23.69,113.28
1,106,99.67,2.43,104.45,14.19,7.32,4.26,services,low,high,low,21.89,85.26
2,126,119.53,2.61,145.95,15.30,8.01,4.13,craftsmanship,low,moderate,very_high,28.00,122.44
3,179,122.99,2.22,112.76,16.01,7.81,4.65,food_production,high,very_high,high,21.05,122.17
4,140,105.63,2.47,112.13,13.92,7.23,3.92,craftsmanship,low,moderate,moderate,36.38,102.84
5,93,134.50,2.60,141.16,10.70,7.41,4.63,services,moderate,very_high,very_high,22.87,134.32
6,100,150.34,2.31,145.67,14.01,7.47,4.76,manufacturing,low,very_high,very_low,31.40,123.93
7,138,102.98,2.30,108.00,16.29,8.03,4.39,services,very_low,high,very_high,25.66,130.74
8,154,120.88,2.49,110.70,13.16,7.70,3.40,services,low,moderate,very_high,30.50,144.19
9,102,107.54,2.34,111.38,13.93,7.41,4.43,food_production,moderate,high,high,39.33,102.95


### Missing values

In [3]:
X_learn.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   age                1000 non-null   int64  
 1   blood_pressure     1000 non-null   float64
 2   calcium            1000 non-null   float64
 3   cholesterol        1000 non-null   float64
 4   hemoglobin         1000 non-null   float64
 5   height             1000 non-null   float64
 6   potassium          1000 non-null   float64
 7   profession         1000 non-null   object 
 8   sarsaparilla       1000 non-null   object 
 9   smurfberry_liquor  1000 non-null   object 
 10  smurfin_donuts     1000 non-null   object 
 11  vitamin_D          1000 non-null   float64
 12  weight             1000 non-null   float64
dtypes: float64(8), int64(1), object(4)
memory usage: 101.7+ KB


### Categorical variables

- Strategy A: One-Hot for *Profession*, Ordinal for the rest
- Strategy B: One-Hot for all four variables

In [4]:
df_combined = X_learn.copy()
df_combined["target"] = y_learn

cat_cols = [
    "profession",
    "sarsaparilla",
    "smurfberry_liquor",
    "smurfin_donuts"
]

for col in cat_cols:
    display(
        df_combined.groupby(col)["target"].agg(
            ["count", "mean", "median"]
        )
    )

,count,mean,median
profession,,,
administration_and_governance,93,0.079140,0.06
craftsmanship,92,0.068478,0.05
food_production,272,0.076397,0.05
manufacturing,179,0.091173,0.07
resource_extraction,171,0.081930,0.06
services,192,0.082344,0.06


,count,mean,median
sarsaparilla,,,
high,149,0.073490,0.050
low,349,0.082636,0.060
moderate,167,0.085868,0.060
very_high,118,0.088051,0.075
very_low,216,0.074352,0.050


,count,mean,median
smurfberry_liquor,,,
high,224,0.084241,0.07
low,141,0.082270,0.06
moderate,377,0.083050,0.06
very_high,165,0.075939,0.05
very_low,92,0.068152,0.05


,count,mean,median
smurfin_donuts,,,
high,221,0.076425,0.05
low,201,0.080796,0.06
moderate,220,0.085136,0.06
very_high,187,0.081979,0.06
very_low,170,0.078765,0.06


In [5]:
X_learn_A = X_learn.copy()
X_test_A = X_test.copy()

consumption_map = {
    "very_low": 0,
    "low": 1,
    "moderate": 2,
    "high": 3,
    "very_high": 4
}

ordinal_cols = [
    "sarsaparilla",
    "smurfberry_liquor",
    "smurfin_donuts"
]

for col in ordinal_cols:
    X_learn_A[col] = X_learn_A[col].replace(consumption_map)
    X_test_A[col] = X_test_A[col].replace(consumption_map)

X_learn_A = pd.get_dummies(X_learn_A, columns=["profession"], dtype=int)
X_test_A = pd.get_dummies(X_test_A, columns=["profession"], dtype=int)

X_test_A = X_test_A.reindex(columns=X_learn_A.columns, fill_value=0)

X_learn_A.head(10)

,age,blood_pressure,calcium,cholesterol,hemoglobin,height,potassium,sarsaparilla,smurfberry_liquor,smurfin_donuts,vitamin_D,weight,profession_administration_and_governance,profession_craftsmanship,profession_food_production,profession_manufacturing,profession_resource_extraction,profession_services
0,161,104.55,2.53,116.47,16.55,7.80,3.98,3,4,0,23.69,113.28,0,0,0,0,0,1
1,106,99.67,2.43,104.45,14.19,7.32,4.26,1,3,1,21.89,85.26,0,0,0,0,0,1
2,126,119.53,2.61,145.95,15.30,8.01,4.13,1,2,4,28.00,122.44,0,1,0,0,0,0
3,179,122.99,2.22,112.76,16.01,7.81,4.65,3,4,3,21.05,122.17,0,0,1,0,0,0
4,140,105.63,2.47,112.13,13.92,7.23,3.92,1,2,2,36.38,102.84,0,1,0,0,0,0
5,93,134.50,2.60,141.16,10.70,7.41,4.63,2,4,4,22.87,134.32,0,0,0,0,0,1
6,100,150.34,2.31,145.67,14.01,7.47,4.76,1,4,0,31.40,123.93,0,0,0,1,0,0
7,138,102.98,2.30,108.00,16.29,8.03,4.39,0,3,4,25.66,130.74,0,0,0,0,0,1
8,154,120.88,2.49,110.70,13.16,7.70,3.40,1,2,4,30.50,144.19,0,0,0,0,0,1
9,102,107.54,2.34,111.38,13.93,7.41,4.43,2,3,3,39.33,102.95,0,0,1,0,0,0


In [6]:
X_learn_B = X_learn.copy()
X_test_B = X_test.copy()

X_learn_B = pd.get_dummies(X_learn_B, columns=cat_cols, dtype=int)
X_test_B = pd.get_dummies(X_test_B, columns=cat_cols, dtype=int)

X_test_B = X_test_B.reindex(columns=X_learn_B.columns, fill_value=0)

X_learn_B.head(10)

,age,blood_pressure,calcium,cholesterol,hemoglobin,height,potassium,vitamin_D,weight,profession_administration_and_governance,...,smurfberry_liquor_high,smurfberry_liquor_low,smurfberry_liquor_moderate,smurfberry_liquor_very_high,smurfberry_liquor_very_low,smurfin_donuts_high,smurfin_donuts_low,smurfin_donuts_moderate,smurfin_donuts_very_high,smurfin_donuts_very_low
0,161,104.55,2.53,116.47,16.55,7.80,3.98,23.69,113.28,0,...,0,0,0,1,0,0,0,0,0,1
1,106,99.67,2.43,104.45,14.19,7.32,4.26,21.89,85.26,0,...,1,0,0,0,0,0,1,0,0,0
2,126,119.53,2.61,145.95,15.30,8.01,4.13,28.00,122.44,0,...,0,0,1,0,0,0,0,0,1,0
3,179,122.99,2.22,112.76,16.01,7.81,4.65,21.05,122.17,0,...,0,0,0,1,0,1,0,0,0,0
4,140,105.63,2.47,112.13,13.92,7.23,3.92,36.38,102.84,0,...,0,0,1,0,0,0,0,1,0,0
5,93,134.50,2.60,141.16,10.70,7.41,4.63,22.87,134.32,0,...,0,0,0,1,0,0,0,0,1,0
6,100,150.34,2.31,145.67,14.01,7.47,4.76,31.40,123.93,0,...,0,0,0,1,0,0,0,0,0,1
7,138,102.98,2.30,108.00,16.29,8.03,4.39,25.66,130.74,0,...,1,0,0,0,0,0,0,0,1,0
8,154,120.88,2.49,110.70,13.16,7.70,3.40,30.50,144.19,0,...,0,0,1,0,0,0,0,0,1,0
9,102,107.54,2.34,111.38,13.93,7.41,4.43,39.33,102.95,0,...,1,0,0,0,0,1,0,0,0,0


### Outliers

In [7]:
X_learn.describe(percentiles=[0.05,0.95])

,age,blood_pressure,calcium,cholesterol,hemoglobin,height,potassium,vitamin_D,weight
count,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,119.885000,109.046440,2.454470,112.437600,14.222370,7.580640,4.286010,29.757290,105.459500
std,29.433078,11.131466,0.207102,23.076467,1.923225,0.403022,0.484628,5.045444,15.830449
min,26.000000,88.030000,1.780000,39.370000,8.020000,6.360000,2.780000,11.390000,60.130000
5%,73.000000,95.397500,2.119500,72.619000,11.098500,6.930000,3.449500,21.429500,81.018500
50%,119.500000,106.580000,2.450000,111.770000,14.215000,7.570000,4.270000,29.955000,104.175000
95%,170.050000,129.527000,2.810000,149.992500,17.260000,8.210000,5.090000,38.010000,132.830000
max,180.000000,163.620000,3.110000,189.060000,20.040000,8.930000,5.570000,48.610000,151.350000


### Standardization

In [15]:
numeric_cols = [
    "age", "blood_pressure", "calcium",
    "cholesterol", "hemoglobin", "height",
    "potassium", "vitamin_D", "weight"
]

cols_A = numeric_cols + ordinal_cols
scaler_A = StandardScaler()
scaler_A.fit(X_learn_A[cols_A])
X_learn_A[cols_A] = scaler_A.transform(X_learn_A[cols_A])
X_test_A[cols_A] = scaler_A.transform(X_test_A[cols_A])

cols_B = numeric_cols
scaler_B = StandardScaler()
scaler_B.fit(X_learn_B[cols_B])
X_learn_B[cols_B] = scaler_B.transform(X_learn_B[cols_B])
X_test_B[cols_B] = scaler_B.transform(X_test_B[cols_B])

X_learn_A.head(10)


,age,blood_pressure,calcium,cholesterol,hemoglobin,height,potassium,sarsaparilla,smurfberry_liquor,smurfin_donuts,vitamin_D,weight,profession_administration_and_governance,profession_craftsmanship,profession_food_production,profession_manufacturing,profession_resource_extraction,profession_services
0,1.397597,-0.404142,0.364881,0.174828,1.210880,0.544560,-0.631748,1.078647,1.529592,-1.510134,-1.203130,0.494263,0,0,0,0,0,1
1,-0.471984,-0.842758,-0.118213,-0.346309,-0.016840,-0.647038,-0.053697,-0.465586,0.665904,-0.774560,-1.560066,-1.276629,0,0,0,0,0,1
2,0.207863,0.942266,0.751357,1.452959,0.560605,1.065885,-0.322078,-0.465586,-0.197785,1.432163,-0.348467,1.073185,0,1,0,0,0,0
3,2.009460,1.253252,-1.132712,0.013978,0.929961,0.569385,0.751446,1.078647,1.529592,0.696589,-1.726636,1.056120,0,0,1,0,0,0
4,0.683757,-0.307071,0.075025,-0.013336,-0.157299,-0.870463,-0.755616,-0.465586,-0.197785,-0.038985,1.313269,-0.165555,0,1,0,0,0,0
5,-0.913885,2.287776,0.703047,1.245285,-1.832408,-0.423613,0.710157,0.306530,1.529592,1.432163,-1.365734,1.824013,0,0,0,0,0,1
6,-0.675938,3.711481,-0.697927,1.440820,-0.110479,-0.274663,0.978538,-0.465586,1.529592,-1.510134,0.325746,1.167354,0,0,0,1,0,0
7,0.615772,-0.545254,-0.746236,-0.192396,1.075623,1.115535,0.214684,-1.237703,0.665904,1.432163,-0.812484,1.597753,0,0,0,0,0,1
8,1.159650,1.063605,0.171643,-0.075335,-0.552666,0.296311,-1.829140,-0.465586,-0.197785,1.432163,0.147278,2.447807,0,0,0,0,0,1
9,-0.607954,-0.135399,-0.552998,-0.045853,-0.152097,-0.423613,0.297263,0.306530,0.665904,0.696589,1.898247,-0.158603,0,0,1,0,0,0


### Feature Selection